# Weibull Accelerated Failure Time 

El **Weibull Accelerated Failure Time (AFT)** es un modelo paramétrico de
análisis de supervivencia que modela directamente el logaritmo del tiempo
de fallo como función lineal de las covariables:

    log(T) = Xβ + σε,  ε ~ Weibull

A diferencia de Cox PH que modela el hazard ratio relativo h(t|X)/h₀(t),
AFT modela el tiempo de fallo absoluto — las covariables "aceleran" o
"desaceleran" el proceso de degradación multiplicando el tiempo de vida
por exp(-β'X). Esto hace que AFT sea más interpretable que Cox en contextos
de ingeniería: un coeficiente βⱼ negativo significa que la componente PCA j
acelera el fallo.

**¿Por qué Weibull AFT para RUL?**
AFT fue evaluado como alternativa a Cox PH tras documentar la inestabilidad
numérica de los modelos de frailty. La distribución Weibull es el modelo
paramétrico estándar para tiempos de fallo en sistemas mecánicos — su
función de hazard monótonamente creciente (β_forma > 1) es coherente con
la degradación progresiva de motores turbofan. Adicionalmente, AFT con
full likelihood puede capturar información distribucional que Cox PH ignora
al usar solo partial likelihood.

**Abordaje en este proyecto**
Se usa `WeibullAFTFitter` de lifelines. t_stop de cada ventana se usa como
duración. RUL se estima via la curva de muerte F(t|X) = 1 - S(t|X):
el primer t* donde F(t*) = confidence_threshold define el ciclo de fallo
predicho, y RUL = t* - t_stop_actual.

**Hiperparámetros considerados**

| Hiperparámetro | Rango explorado | Justificación |
|----------------|----------------|---------------|
| `feature_set` | A, B, C, D | Evalúa si features adicionales mejoran la estimación Weibull |
| `window_size` | 20, 25, 30 | Horizonte temporal de cada ventana |
| `n_components` | 10, 15, 20 | Dimensionalidad del espacio de covariables |
| `clipping_threshold` | 115, 120, 125 | Techo del RUL predicho |
| `confidence_threshold` | 0.3, 0.5, 0.95 | Umbral de F(t) para declarar fallo predicho |
| `penalizer` | 0.0, 0.1, 1.0 | Regularización elastic net sobre β |
| `l1_ratio` | 0.0, 0.5, 1.0 | Mixing L1/L2 — activo cuando penalizer > 0 |
| `fit_intercept` | True, False | Intercepto en el predictor lineal de escala Weibull |

## Resultados GGS y diagnóstico

**Resumen del GGS**
- Configuraciones evaluadas: 5,832
- Exitosas: 1,992 (34.2%) — Fallidas: 3,840 (65.8%)
- Folds: 5 (GroupKFold por motor)

**Top 10 configuraciones**

| feature_set | window_size | n_components | clipping_threshold | conf_thresh | penalizer | l1_ratio | fit_intercept | S-Score | MAE | RMSE | C-Index |
|-------------|-------------|--------------|-------------------|-------------|-----------|----------|---------------|---------|-----|------|---------|
| D | 20 | 10 | 115 | 0.3 | 1.0 | 0.5 | True | 209.8 | 38.5 | 47.2 | 0.855 |
| C | 30 | 15 | 115 | 0.3 | 1.0 | 1.0 | True | 224.6 | 40.9 | 48.8 | 0.841 |
| D | 20 | 10 | 120 | 0.3 | 1.0 | 0.5 | True | 259.4 | 40.2 | 48.7 | 0.851 |
| C | 20 | 10 | 115 | 0.3 | 1.0 | 1.0 | True | 277.3 | 39.5 | 48.6 | 0.844 |
| C | 30 | 15 | 120 | 0.3 | 1.0 | 1.0 | True | 277.7 | 42.9 | 50.4 | 0.836 |
| C | 25 | 10 | 115 | 0.3 | 1.0 | 1.0 | True | 286.8 | 40.6 | 49.4 | 0.837 |
| B | 30 | 15 | 115 | 0.3 | 1.0 | 1.0 | True | 290.8 | 41.0 | 49.7 | 0.826 |
| C | 30 | 20 | 115 | 0.3 | 1.0 | 1.0 | True | 290.8 | 41.0 | 49.7 | 0.826 |
| C | 25 | 20 | 115 | 0.3 | 1.0 | 1.0 | True | 306.1 | 40.3 | 49.5 | 0.831 |
| C | 20 | 20 | 115 | 0.3 | 1.0 | 1.0 | True | 308.9 | 40.0 | 49.4 | 0.835 |

### Diagnóstico

**Tasa de fallos y causa**

El 65.8% de configuraciones fallidas responde a la misma causa estructural
que CoxPH — tasa de eventos de 0.59% — pero con un agravante adicional:
Weibull AFT usa **full likelihood** en lugar de partial likelihood, lo que
requiere estimar simultáneamente β, la forma ρ y la escala σ de la
distribución. Con tan pocos eventos, el optimizador diverge con más
frecuencia que Cox. La regularización fuerte (`penalizer=1.0`, exclusiva
en el top 10) es la única forma de estabilizar la estimación — a costa de
shrinkear β hacia cero.

**El dato más relevante: C-Index = 0.855**

AFT produce el C-Index más alto de los modelos de supervivencia y
sorprendentemente competitivo con los modelos de regresión (SVR=0.915,
RF=0.909). Esto revela que AFT discrimina el ranking de degradación con
cierta precisión — sabe *quién* está más degradado — pero sus predicciones
absolutas de RUL son pobres (MAE=38.5) porque β≈0 bajo regularización
fuerte colapsa las predicciones hacia la mediana de la distribución Weibull.

La separación entre C-Index competitivo y MAE/RMSE inaceptables es la
firma característica de un modelo cuya capacidad discriminativa es real
pero cuya calibración absoluta está comprometida por la escasez de eventos.

**Diferencias clave respecto a CoxPH**

AFT prefiere `feature_set=C` y `D` — completamente opuesto a los modelos
de regresión que preferían `A`. La full likelihood de Weibull necesita más
información distribucional para estimar la forma y escala — las features
de memoria (C) y frecuencia (D) aportan información sobre la distribución
temporal de la degradación que el set básico `A` no captura. Este es el
único modelo del proyecto donde `feature_set=A` no es óptimo.

`penalizer=1.0` domina en AFT vs `penalizer=0.1` en Cox — la full
likelihood es más inestable que la partial likelihood ante escasez de
eventos y requiere regularización más agresiva para converger.

**La incompatibilidad es estructural e independiente de las features**

Igual que CoxPH, el problema no es la representación de covariables sino
la naturaleza del dataset. Con sensores crudos la tasa de eventos sería
idéntica — un evento por motor por definición en C-MAPSS FD001:

```
n_eventos / n_filas = 140 / 23,800 ≈ 0.59%  — invariante
```

La tensión fundamental entre formato fila-por-ciclo (necesario para
explotar los sensores) y tasa de eventos suficiente (necesaria para
los modelos de supervivencia) es irresoluble para este dataset.

### Comparación acumulada

| Métrica | NB | DT | RF | SVR | XGB | CoxPH | **AFT** |
|---------|----|----|-----|-----|-----|-------|---------|
| S-Score | 2.447 | 2.755 | 1.883 | 1.825 | 1.782 | 7,182 | **210** |
| MAE | 10.39 | 7.79 | 6.98 | 6.75 | 7.10 | 32.5 | **38.5** |
| RMSE | 13.36 | 12.22 | 10.62 | 10.24 | 10.45 | 49.4 | **47.2** |
| C-Index | 0.908 | 0.893 | 0.909 | 0.915 | 0.912 | 0.544 | **0.855** |

AFT supera a CoxPH en todas las métricas y produce un C-Index
sorprendentemente competitivo (0.855). Sin embargo, las predicciones
absolutas de RUL son inaceptables para uso en producción — MAE de
38.5 ciclos representa un error del ~33% sobre el rango útil de
predicción (0-115 ciclos).

### Conclusión

Weibull AFT es el modelo de supervivencia más informativo del proyecto —
su C-Index de 0.855 sugiere que con una tasa de eventos mayor podría
ser un modelo serio. Sin embargo, comparte la incompatibilidad estructural
de CoxPH con el pipeline de ventanas deslizantes: la tasa de eventos de
0.59% hace inviable la estimación estable de los parámetros Weibull sin
regularización agresiva que compromete la calibración absoluta.

**No se seleccionan hiperparámetros de producción.** Weibull AFT queda
documentado como resultado negativo con matiz: a diferencia de CoxPH
(C-Index≈aleatorio), AFT demuestra capacidad discriminativa real que
no puede traducirse en predicciones precisas de RUL por la escasez
estructural de eventos en C-MAPSS FD001.

In [2]:
import pandas as pd

results_path = 'outputs/ggs/results/WeibullAFTModel_dd4bcf5e_20260512_1116.csv'

cols_params = ['feature_set', 'window_size', 'n_components', 'clipping_threshold', 
               'confidence_threshold', 'penalizer', 'l1_ratio', 'fit_intercept', 
               'mean_S_score', 'mean_C_index', 'mean_MAE', 'mean_RMSE']

df_results = pd.read_csv(results_path)

total = len(df_results)
exitosas = df_results['mean_S_score'].notna().sum()
fallidas = df_results['mean_S_score'].isna().sum()
n_duplicados = df_results.duplicated(subset=cols_params).sum()

print(f"Total configuraciones: {total}")
print(f"Exitosas:              {exitosas}")
print(f"Fallidas (NaN):        {fallidas}")
print(f"Duplicados:            {n_duplicados}")
print(f"Únicas:                {total - n_duplicados}")

Total configuraciones: 5832
Exitosas:              1992
Fallidas (NaN):        3840
Duplicados:            0
Únicas:                5832


In [3]:
df_top = (
    df_results
    .dropna(subset=['mean_S_score'])
    .sort_values('mean_S_score', ascending=True)
    .head(10)
    .reset_index(drop=True)
)

df_top[cols_params]

,feature_set,window_size,n_components,clipping_threshold,confidence_threshold,penalizer,l1_ratio,fit_intercept,mean_S_score,mean_C_index,mean_MAE,mean_RMSE
0,D,20,10,115,0.3,1.0,0.5,True,209.778083,0.855188,38.539443,47.167833
1,C,30,15,115,0.3,1.0,1.0,True,224.588502,0.840613,40.949924,48.788670
2,D,20,10,120,0.3,1.0,0.5,True,259.351601,0.851152,40.236892,48.688513
3,C,20,10,115,0.3,1.0,1.0,True,277.323689,0.844367,39.500991,48.619324
4,C,30,15,120,0.3,1.0,1.0,True,277.699692,0.835792,42.909206,50.376978
5,C,25,10,115,0.3,1.0,1.0,True,286.776110,0.837004,40.612826,49.426286
6,B,30,15,115,0.3,1.0,1.0,True,290.787187,0.825507,41.020210,49.670462
7,C,30,20,115,0.3,1.0,1.0,True,290.787187,0.825507,41.020210,49.670462
8,C,25,20,115,0.3,1.0,1.0,True,306.116283,0.831109,40.338076,49.512389
9,C,20,20,115,0.3,1.0,1.0,True,308.907711,0.835247,39.966327,49.388695
